In [64]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [65]:
!pip install ultralytics

In [66]:
import os
import subprocess

video_dir = "/content/drive/MyDrive/Drone/input_video"
frames_root = "/content/drive/MyDrive/Drone/frames"

fps = 5

for filename in os.listdir(video_dir):
    if filename.lower().endswith(".mp4"):
        video_path = os.path.join(video_dir, filename)

        video_name = os.path.splitext(filename)[0]
        output_folder = os.path.join(frames_root, video_name)
        os.makedirs(output_folder, exist_ok=True)

        output_pattern = os.path.join(output_folder, "frame_%04d.jpg")

        cmd = [
            "ffmpeg",
            "-i", video_path,
            "-vf", f"fps={fps}",
            output_pattern
        ]

        print(f"Processing: {filename}")
        subprocess.run(cmd)

Processing: drone_video_2.mp4
Processing: drone_video_1.mp4


In [67]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")
#model = YOLO("/content/drive/MyDrive/Drone/yolov26n_drone_set/weights/best.pt")

In [4]:
results = model.train(
    data="/content/drive/MyDrive/Drone/data.yaml",
    epochs=20,
    imgsz=640,
    project="/content/drive/MyDrive/Drone",
    name="yolov26n_drone_set",
)

Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Drone/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov26n_drone_set, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_m

In [69]:
import os
import cv2
import pandas as pd

frames_root = "/content/drive/MyDrive/Drone/frames"
output_folder = "/content/drive/MyDrive/Drone/detections"
os.makedirs(output_folder, exist_ok=True)

detections_data = []

frame_dirs = sorted([
    d for d in os.listdir(frames_root)
    if os.path.isdir(os.path.join(frames_root, d))
])

for video_id, frame_dir_name in enumerate(frame_dirs, start=1):
    frame_dir_path = os.path.join(frames_root, frame_dir_name)

    print(f"Processing video_id={video_id}, frames folder={frame_dir_name}")

    results = model.predict(
        source=frame_dir_path,
        save=False,
        stream=True,
        conf=0.15
    )

    for i, res in enumerate(results):
        if len(res.boxes) > 0:
            annotated_frame = res.orig_img.copy()
            filename = f"{frame_dir_name}_frame_{i:04d}.jpg"
            filepath = os.path.join(output_folder, filename)

            for box in res.boxes:
                class_index = int(box.cls[0])
                class_label = model.names[class_index]
                confidence = float(box.conf[0])
                coords = [int(c) for c in box.xyxy[0].tolist()]

                cv2.rectangle(
                    annotated_frame,
                    (coords[0], coords[1]),
                    (coords[2], coords[3]),
                    (0, 0, 255),
                    2
                )

                label = f"{class_label} {confidence:.2f}"
                cv2.putText(
                    annotated_frame,
                    label,
                    (coords[0], max(coords[1] - 10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (0, 0, 255),
                    2
                )

                detections_data.append({
                    "video_id": video_id,
                    "frame_directory": frame_dir_name,
                    "frame_index": i,
                    "file_name": filename,
                    "file_path": filepath,
                    "class_label": class_label,
                    "bounding_box": coords,
                    "confidence_score": confidence
                })

            cv2.imwrite(filepath, annotated_frame)

df = pd.DataFrame(detections_data)
if not df.empty:
    df['center_x'] = df['bounding_box'].apply(lambda b: (b[0] + b[2]) / 2)
    df['center_y'] = df['bounding_box'].apply(lambda b: (b[1] + b[3]) / 2)

Processing video_id=1, frames folder=drone_video_1

image 1/828 /content/drive/MyDrive/Drone/frames/drone_video_1/frame_0001.jpg: 384x640 2 drones, 12.8ms
image 2/828 /content/drive/MyDrive/Drone/frames/drone_video_1/frame_0002.jpg: 384x640 1 drone, 14.0ms
image 3/828 /content/drive/MyDrive/Drone/frames/drone_video_1/frame_0003.jpg: 384x640 1 drone, 11.2ms
image 4/828 /content/drive/MyDrive/Drone/frames/drone_video_1/frame_0004.jpg: 384x640 1 drone, 11.8ms
image 5/828 /content/drive/MyDrive/Drone/frames/drone_video_1/frame_0005.jpg: 384x640 1 drone, 11.3ms
image 6/828 /content/drive/MyDrive/Drone/frames/drone_video_1/frame_0006.jpg: 384x640 1 drone, 11.4ms
image 7/828 /content/drive/MyDrive/Drone/frames/drone_video_1/frame_0007.jpg: 384x640 1 drone, 11.4ms
image 8/828 /content/drive/MyDrive/Drone/frames/drone_video_1/frame_0008.jpg: 384x640 2 drones, 11.4ms
image 9/828 /content/drive/MyDrive/Drone/frames/drone_video_1/frame_0009.jpg: 384x640 2 drones, 16.8ms
image 10/828 /content/drive

In [70]:
df

,video_id,frame_directory,frame_index,file_name,file_path,class_label,bounding_box,confidence_score,center_x,center_y
0,1,drone_video_1,0,drone_video_1_frame_0000.jpg,/content/drive/MyDrive/Drone/detections/drone_...,drone,"[844, 165, 924, 246]",0.387733,884.0,205.5
1,1,drone_video_1,0,drone_video_1_frame_0000.jpg,/content/drive/MyDrive/Drone/detections/drone_...,drone,"[845, 169, 917, 244]",0.318391,881.0,206.5
2,1,drone_video_1,1,drone_video_1_frame_0001.jpg,/content/drive/MyDrive/Drone/detections/drone_...,drone,"[847, 172, 916, 248]",0.803412,881.5,210.0
3,1,drone_video_1,2,drone_video_1_frame_0002.jpg,/content/drive/MyDrive/Drone/detections/drone_...,drone,"[841, 175, 921, 253]",0.865342,881.0,214.0
4,1,drone_video_1,3,drone_video_1_frame_0003.jpg,/content/drive/MyDrive/Drone/detections/drone_...,drone,"[840, 176, 918, 257]",0.865357,879.0,216.5
...,...,...,...,...,...,...,...,...,...,...
604,2,drone_video_2,1481,drone_video_2_frame_1481.jpg,/content/drive/MyDrive/Drone/detections/drone_...,drone,"[1743, 29, 1784, 75]",0.162032,1763.5,52.0
605,2,drone_video_2,2069,drone_video_2_frame_2069.jpg,/content/drive/MyDrive/Drone/detections/drone_...,drone,"[1099, 232, 1130, 280]",0.466514,1114.5,256.0
606,2,drone_video_2,2070,drone_video_2_frame_2070.jpg,/content/drive/MyDrive/Drone/detections/drone_...,drone,"[1055, 202, 1086, 245]",0.177512,1070.5,223.5
607,2,drone_video_2,2071,drone_video_2_frame_2071.jpg,/content/drive/MyDrive/Drone/detections/drone_...,drone,"[1023, 176, 1055, 220]",0.191192,1039.0,198.0


In [71]:
!pip install filterpy

In [75]:
from filterpy.kalman import KalmanFilter
import numpy as np

def create_drone_tracker():
    kf = KalmanFilter(dim_x=4, dim_z=2)
    dt = 1.0

    kf.F = np.array([[1, 0, dt, 0],
                     [0, 1, 0, dt],
                     [0, 0, 1, 0],
                     [0, 0, 0, 1]])

    kf.H = np.array([[1, 0, 0, 0],
                     [0, 1, 0, 0]])

    kf.P *= 1000.
    kf.R = 5
    kf.Q = 0.1

    return kf

In [76]:
import cv2
import os
import numpy as np

def track_drone_filtered_frames(frame_folder, detections_df, output_video_name, fps=5):
    images = sorted([img for img in os.listdir(frame_folder) if img.endswith(".jpg")])
    if not images:
        print(f"No images in {frame_folder}")
        return

    first_frame = cv2.imread(os.path.join(frame_folder, images[0]))
    h, w, _ = first_frame.shape

    out = cv2.VideoWriter(output_video_name, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

    tracker = create_drone_tracker()
    initialized = False
    trajectory = []

    for i, img_name in enumerate(images):
        frame = cv2.imread(os.path.join(frame_folder, img_name))

        current_det = detections_df[detections_df['frame_index'] == i]

        tracker.predict()

        if not current_det.empty:
            best_det = current_det.iloc[0]
            z = np.array([[best_det['center_x']], [best_det['center_y']]])

            if not initialized:
                tracker.x[:2] = z
                initialized = True
            else:
                tracker.update(z)

            bbox = best_det['bounding_box']
            cv2.rectangle(frame, (bbox[0], bbox[1]), (bbox[2], bbox[3]), (0, 0, 255), 2)

            curr_x, curr_y = int(tracker.x[0]), int(tracker.x[1])
            trajectory.append((curr_x, curr_y))

            if len(trajectory) > 1:
                for j in range(1, len(trajectory)):
                    cv2.line(frame, trajectory[j-1], trajectory[j], (0, 255, 0), 2)

            out.write(frame)

    out.release()
    print(f"Filtered video saved to: {output_video_name}")

In [77]:
import os

frames_root = "/content/drive/MyDrive/Drone/frames"
output_root = "/content/drive/MyDrive/Drone"

frame_dirs = sorted([
    d for d in os.listdir(frames_root)
    if os.path.isdir(os.path.join(frames_root, d))
])

for video_id, frame_dir_name in enumerate(frame_dirs, start=1):
    frame_folder = os.path.join(frames_root, frame_dir_name)

    detections_for_video = df[df["video_id"] == video_id]

    if detections_for_video.empty:
        print(f"Skipping video_id={video_id}, folder={frame_dir_name} (no detections)")
        continue

    output_video_name = os.path.join(
        output_root,
        f"tracking_video_{video_id}.mp4"
    )

    print(f"Tracking video_id={video_id}, folder={frame_dir_name}")

    track_drone_filtered_frames(
        frame_folder=frame_folder,
        detections_df=detections_for_video,
        output_video_name=output_video_name
    )

Tracking video_id=1, folder=drone_video_1


/tmp/ipykernel_3484/1641037391.py:40: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  curr_x, curr_y = int(tracker.x[0]), int(tracker.x[1])


Filtered video saved to: /content/drive/MyDrive/Drone/tracking_video_1.mp4
Tracking video_id=2, folder=drone_video_2
Filtered video saved to: /content/drive/MyDrive/Drone/tracking_video_2.mp4


In [78]:
!pip install datasets huggingface_hub

In [89]:
from datasets import Dataset, Image as HFImage
import pandas as pd
import os

base_path = "/content/drive/MyDrive/Drone/detections/"
df['image'] = df['file_path'].apply(lambda x: os.path.join(base_path, x))

hf_dataset = Dataset.from_pandas(df)

hf_dataset = hf_dataset.cast_column("image", HFImage())

print(hf_dataset)
print(hf_dataset[0])

Dataset({
    features: ['video_id', 'frame_directory', 'frame_index', 'file_name', 'file_path', 'class_label', 'bounding_box', 'confidence_score', 'center_x', 'center_y', 'image'],
    num_rows: 609
})
{'video_id': 1, 'frame_directory': 'drone_video_1', 'frame_index': 0, 'file_name': 'drone_video_1_frame_0000.jpg', 'file_path': '/content/drive/MyDrive/Drone/detections/drone_video_1_frame_0000.jpg', 'class_label': 'drone', 'bounding_box': [844, 165, 924, 246], 'confidence_score': 0.38773313164711, 'center_x': 884.0, 'center_y': 205.5, 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1920x1080 at 0x7A57DC286300>}


In [93]:
from huggingface_hub import notebook_login
notebook_login()

hf_dataset.push_to_hub("shreyonroy/Assignment3_Parquet_Drone_Detections")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/609 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 34.2MB / 34.2MB            

CommitInfo(commit_url='https://huggingface.co/datasets/shreyonroy/Assignment3_Parquet_Drone_Detections/commit/2c60e05c76f8567ca1f090cbf75e8ba27472fd60', commit_message='Upload dataset', commit_description='', oid='2c60e05c76f8567ca1f090cbf75e8ba27472fd60', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/shreyonroy/Assignment3_Parquet_Drone_Detections', endpoint='https://huggingface.co', repo_type='dataset', repo_id='shreyonroy/Assignment3_Parquet_Drone_Detections'), pr_revision=None, pr_num=None)